# LeetCode 1087 — Brace Expansion bằng Python

Link đề: https://leetcode.com/problems/brace-expansion/ · Độ khó: **Medium** · Premium (cần tài khoản LeetCode Premium để xem đề) · Chủ đề: String (chuỗi), Backtracking (quay lui), Breadth-First Search, Sorting (sắp xếp)

> **Lưu ý:** đề bài của bài Premium này được viết theo hiểu biết và theo bản mô tả đã có; nên đối chiếu lại nếu bạn có tài khoản Premium.

Skeleton LeetCode:

```python
class Solution:
    def expand(self, s: str) -> List[str]:
        ...
```

Ghi chú: chuỗi `s` biểu diễn một danh sách các từ. Mỗi vị trí chữ cái trong từ có một hay nhiều lựa chọn: nếu chỉ có một lựa chọn thì chữ cái được viết nguyên; nếu có nhiều hơn một lựa chọn thì chúng nằm trong cặp ngoặc nhọn, ví dụ `"{a,b,c}"` là các lựa chọn `["a", "b", "c"]`. Ví dụ `s = "a{b,c}"`: ký tự đầu luôn là `'a'`, ký tự thứ hai là `'b'` hoặc `'c'`, danh sách là `["ab", "ac"]`. Trả về **mọi** từ có thể tạo ra, **sắp xếp theo thứ tự từ điển**. Ý tưởng chính: tách `s` thành các **nhóm lựa chọn** (mỗi chữ cái đứng riêng là nhóm một phần tử, mỗi cặp ngoặc là một nhóm nhiều phần tử, **sắp xếp** lựa chọn của mỗi nhóm), rồi sinh **tích Descartes** bằng quay lui theo thứ tự các nhóm; vì mọi từ có cùng độ dài và các lựa chọn đã sắp xếp, kết quả tự động đã đúng thứ tự từ điển.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chuỗi là **ghép** của các nhóm lựa chọn, và đáp án là **tích Descartes** các nhóm, sắp xếp từ điển.
2. Nắm các công cụ Python cần thiết: duyệt chuỗi theo chỉ số, `str.find`, `str.split(",")`, `sorted`, `itertools.product`, quay lui bằng đệ quy, hàng đợi / danh sách để xây theo từng tầng.
3. Hiểu vì sao nếu các lựa chọn của mỗi nhóm đã **sắp xếp tăng** và mọi từ **cùng độ dài** thì duyệt tích Descartes theo thứ tự nhóm cho ra kết quả **đã sắp xếp**, không cần `sort` cuối.
4. Đi qua 4 cách: đệ quy trên hậu tố của chuỗi rồi sắp xếp (brute force, dùng làm oracle), phân tích thành nhóm + `itertools.product` + `sorted`, quay lui theo nhóm đã sắp xếp (cách nộp chính) và xây từng tầng bằng danh sách.
5. Tránh các lỗi kinh điển: quên sắp xếp, coi dấu phẩy là một lựa chọn, không bỏ dấu `}`, biểu diễn chữ cái đứng riêng bằng chuỗi rồi đưa vào `product` (chuỗi bị tách thành từng chữ), và ghép ngược thứ tự.

## 2. Đề bài & ví dụ

Cho chuỗi `s` biểu diễn một danh sách các từ. Mỗi chữ cái trong từ có một hoặc nhiều lựa chọn.

- Nếu chỉ có một lựa chọn thì chữ cái được viết nguyên.
- Nếu có nhiều hơn một lựa chọn thì các lựa chọn nằm trong cặp ngoặc nhọn. Ví dụ `"{a,b,c}"` biểu diễn các lựa chọn `["a", "b", "c"]`.

Ví dụ nếu `s = "a{b,c}"`, ký tự đầu tiên luôn là `'a'`, ký tự thứ hai có thể là `'b'` hoặc `'c'`. Danh sách ban đầu là `["ab", "ac"]`.

Trả về tất cả các từ có thể tạo ra theo cách này, **sắp xếp theo thứ tự từ điển**.

**Ví dụ 1:** `s = "{a,b}c{d,e}f"` -> `["acdf","acef","bcdf","bcef"]`.

**Ví dụ 2:** `s = "abcd"` -> `["abcd"]`.

**Ràng buộc:**

- `1 <= s.length <= 50`
- `s` gồm ngoặc nhọn `'{'`, `'}'`, dấu phẩy `','` và chữ cái thường.
- `s` được đảm bảo hợp lệ.
- Không có ngoặc nhọn lồng nhau.
- Mọi ký tự trong một cặp ngoặc nhọn đều khác nhau.

In [1]:
from typing import List
from itertools import product, permutations
from collections import deque
import random, string, time

examples = [
    ("{a,b}c{d,e}f", ["acdf", "acef", "bcdf", "bcef"]),
    ("abcd", ["abcd"]),
]
for s, expected in examples:
    print(f"s={s!r} -> kỳ vọng {expected}")

s='{a,b}c{d,e}f' -> kỳ vọng ['acdf', 'acef', 'bcdf', 'bcef']
s='abcd' -> kỳ vọng ['abcd']


## 3. Kiến thức nền

### 3.1. Duyệt chuỗi, `find` và `split`

`s.find("}", i)` trả về vị trí đầu tiên của `}` từ `i` trở đi (hoặc `-1`). `s[i + 1:j].split(",")` tách phần giữa hai ngoặc thành các lựa chọn.

In [2]:
s = "{a,b}c{d,e}f"
i = 0
j = s.find("}", i)
print(j, s[i + 1:j], s[i + 1:j].split(","))
print("abc,def".split(","), "a".split(","))

4 a,b ['a', 'b']
['abc', 'def'] ['a']


### 3.2. Tích Descartes và `itertools.product`

`product(A, B, C)` sinh mọi bộ `(a, b, c)` theo thứ tự **từ điển của chỉ số**: phần tử cuối thay đổi nhanh nhất. Nếu `A`, `B`, `C` đã sắp xếp tăng, các bộ được sinh theo thứ tự tăng. Lưu ý: nếu truyền một **chuỗi** thì `product` coi nó là dãy các chữ cái.

In [3]:
print(list(product("ab", "c", "de")))
print(["".join(t) for t in product(["a", "b"], ["c"], ["d", "e"])])
print(list(product("abc")))                # chuỗi "abc" bị tách thành các lựa chọn 'a', 'b', 'c'
print(list(product(["abc"])))              # danh sách một chuỗi: một lựa chọn "abc"

[('a', 'c', 'd'), ('a', 'c', 'e'), ('b', 'c', 'd'), ('b', 'c', 'e')]
['acd', 'ace', 'bcd', 'bce']
[('a',), ('b',), ('c',)]
[('abc',)]


### 3.3. Quay lui (backtracking)

Quay lui xây từng phần của lời giải: ở mỗi bước chọn một khả năng, đi sâu, rồi **hoàn tác** lựa chọn để thử khả năng khác. Với tích Descartes: tầng `k` chọn một ký tự trong nhóm `k`; khi tới tầng cuối thì ghi nhận từ.

In [4]:
def tiny_product(groups):
    out, cur = [], []
    def dfs(k):
        if k == len(groups):
            out.append("".join(cur))
            return
        for ch in groups[k]:
            cur.append(ch)          # chọn
            dfs(k + 1)              # đi sâu
            cur.pop()               # hoàn tác
    dfs(0)
    return out
print(tiny_product([["a", "b"], ["c"], ["d", "e"]]))

['acd', 'ace', 'bcd', 'bce']


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 1: `s = "{a,b}c{d,e}f"`.

- Tách thành các nhóm theo thứ tự: `{a,b}`, `c`, `{d,e}`, `f` ⇒ `[["a","b"], ["c"], ["d","e"], ["f"]]`.
- Mỗi từ chọn đúng một ký tự từ mỗi nhóm: `2 * 1 * 2 * 1 = 4` từ.
- Quay lui theo thứ tự nhóm, lựa chọn nhỏ trước: `a-c-d-f`, `a-c-e-f`, `b-c-d-f`, `b-c-e-f` ⇒ `["acdf","acef","bcdf","bcef"]` ✓ đã đúng thứ tự từ điển.

Vì sao không cần sắp xếp cuối? Mọi từ có cùng độ dài (mỗi nhóm cho đúng một chữ cái). So sánh hai từ: vị trí khác nhau đầu tiên quyết định. Quay lui duyệt nhóm đầu tiên ở vòng ngoài nhất với lựa chọn tăng dần, rồi nhóm thứ hai, … nên thứ tự sinh ra chính là thứ tự từ điển.

## 5. Cách 1 — Brute Force: đệ quy trực tiếp trên hậu tố của chuỗi

**Ý tưởng.** Đúng nguyên văn đề, không tách nhóm trước: `expand(s)`:

- Nếu `s` rỗng: trả về `[""]`.
- Nếu `s[0] == "{"`: tìm `}` ở vị trí `j`; các lựa chọn là `s[1:j].split(",")`; kết quả là mọi `o + rest` với `o` là lựa chọn và `rest` thuộc `expand(s[j + 1:])`.
- Ngược lại: `s[0]` là một chữ cái đứng riêng; kết quả là `s[0] + rest` với `rest` thuộc `expand(s[1:])`.

Cuối cùng `sorted`. Vì sinh từ **mọi** khả năng rồi mới sắp xếp, đây là **oracle độc lập**.

In [5]:
def expand_recursive(s: str) -> List[str]:
    def rec(t: str) -> List[str]:
        if not t:
            return [""]
        if t[0] == "{":
            j = t.find("}")
            options = t[1:j].split(",")
            rest = rec(t[j + 1:])
        else:
            options = [t[0]]
            rest = rec(t[1:])
        return [o + r for o in options for r in rest]
    return sorted(rec(s))

for s, expected in examples:
    got = expand_recursive(s)
    print(f"đệ quy hậu tố({s!r}) = {got}", "OK" if got == expected else "SAI")
    assert got == expected

đệ quy hậu tố('{a,b}c{d,e}f') = ['acdf', 'acef', 'bcdf', 'bcef'] OK
đệ quy hậu tố('abcd') = ['abcd'] OK


**Độ phức tạp và nút thắt.** Số từ có thể lên tới vài nghìn (`3^7 = 2187` với `s = "{a,b,c}" * 7`, độ dài `49`), mỗi từ được ghép lại nhiều lần qua các tầng đệ quy `rest`, và còn phải sắp xếp: `O(W * L + W log W)` với `W` là số từ, `L` là độ dài. Đo thử:

In [6]:
for k in (4, 5, 6, 7):
    s = "{a,b,c}" * k
    t0 = time.perf_counter()
    res = expand_recursive(s)
    dt = time.perf_counter() - t0
    print(f"{k} nhóm 3 lựa chọn: {len(res):5} từ, {dt * 1000:7.2f} ms")

4 nhóm 3 lựa chọn:    81 từ,    0.02 ms
5 nhóm 3 lựa chọn:   243 từ,    0.06 ms
6 nhóm 3 lựa chọn:   729 từ,    0.07 ms
7 nhóm 3 lựa chọn:  2187 từ,    0.18 ms


Nút thắt: ta phải sắp xếp lại kết quả mà thực ra có thể sinh **đã đúng thứ tự** nếu duyệt các nhóm theo thứ tự với lựa chọn tăng dần.

## 6. Ý tưởng thuật toán / bước đột phá

**Tách nhóm rồi sinh theo thứ tự.** Quét `s` một lượt: gặp `{` thì đọc đến `}` và lấy các chữ cái (bỏ dấu phẩy) làm một nhóm, **sắp xếp** nhóm đó; gặp chữ cái thường thì tạo nhóm một phần tử. Được danh sách nhóm `groups`. Đáp án là tích Descartes `groups[0] x groups[1] x ...`, duyệt theo thứ tự các nhóm.

**Không cần sắp xếp cuối.** Mọi từ có độ dài bằng số nhóm. Với hai từ khác nhau, xét nhóm đầu tiên mà chúng chọn khác nhau; từ chọn chữ cái nhỏ hơn ở nhóm đó nhỏ hơn. Quay lui duyệt nhóm `0` ở vòng ngoài nhất với các lựa chọn tăng dần, nên các từ ra theo đúng thứ tự từ điển.

| Cách | Ý tưởng |
|---|---|
| 2 | Tách nhóm + `itertools.product` + `sorted` |
| 3 | Tách nhóm, sắp xếp từng nhóm, quay lui theo nhóm (nộp) |
| 4 | Xây theo từng tầng: danh sách từ hiện có ghép với từng lựa chọn của nhóm tiếp theo |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Tách nhóm + `itertools.product`

**Giải thích từng đoạn.**

- `parse(s)`: quét `s`; `{` → đọc đến `}`, lấy các chữ cái cách nhau bởi dấu phẩy; chữ cái thường → nhóm một phần tử (một **danh sách** chứa một chuỗi một ký tự).
- Sắp xếp từng nhóm, rồi `["".join(t) for t in product(*groups)]`.
- `sorted` cuối cùng (bảo đảm, dù đã đúng thứ tự).

In [7]:
def parse_groups(s: str) -> List[List[str]]:
    groups, i = [], 0
    while i < len(s):
        if s[i] == "{":
            j = s.find("}", i)
            groups.append(sorted(s[i + 1:j].split(",")))
            i = j + 1
        else:
            groups.append([s[i]])                 # chữ cái đứng riêng: nhóm một phần tử (DANH SÁCH, không phải chuỗi)
            i += 1
    return groups

def expand_product(s: str) -> List[str]:
    return sorted("".join(t) for t in product(*parse_groups(s)))

print(parse_groups("{a,b}c{d,e}f"))
for s, expected in examples:
    got = expand_product(s)
    print(f"product({s!r}) = {got}", "OK" if got == expected else "SAI")
    assert got == expected

[['a', 'b'], ['c'], ['d', 'e'], ['f']]
product('{a,b}c{d,e}f') = ['acdf', 'acef', 'bcdf', 'bcef'] OK
product('abcd') = ['abcd'] OK


**Code nộp LeetCode (Cách 2):**

In [8]:
class Solution:
    def expand(self, s: str) -> List[str]:
        from itertools import product
        groups = []                                      # mỗi nhóm là danh sách các ký tự lựa chọn
        i = 0
        while i < len(s):
            if s[i] == "{":
                j = s.find("}", i)                       # cặp ngoặc đóng
                groups.append(s[i + 1:j].split(","))
                i = j + 1
            else:
                groups.append([s[i]])                    # chữ cái đứng riêng
                i += 1
        return sorted("".join(t) for t in product(*groups))

sol_product = lambda s: Solution().expand(s)
for s, expected in examples:
    assert sol_product(s) == expected
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** `O(W * L + W log W)` (`W` từ, độ dài `L`), bộ nhớ `O(W * L)`.

---

### 7.2. Cách 3 — Quay lui theo nhóm đã sắp xếp (cách nộp chính)

**Giải thích từng đoạn.**

- Tách nhóm như trên và **sắp xếp** từng nhóm tăng dần.
- `dfs(k)`: nếu `k == len(groups)` thì ghi nhận từ đang xây `"".join(cur)`; ngược lại với mỗi lựa chọn `ch` của `groups[k]` (đã tăng dần): thêm vào `cur`, gọi `dfs(k + 1)`, rồi bỏ ra.
- Kết quả đã đúng thứ tự từ điển (xem mục 6), không cần `sort`.

In [9]:
def expand_backtrack(s: str) -> List[str]:
    groups, i = [], 0
    while i < len(s):
        if s[i] == "{":
            j = s.find("}", i)
            groups.append(sorted(s[i + 1:j].split(",")))
            i = j + 1
        else:
            groups.append([s[i]])
            i += 1
    res, cur = [], []

    def dfs(k: int) -> None:
        if k == len(groups):
            res.append("".join(cur))
            return
        for ch in groups[k]:
            cur.append(ch)
            dfs(k + 1)
            cur.pop()

    dfs(0)
    return res

for s, expected in examples:
    got = expand_backtrack(s)
    print(f"quay lui({s!r}) = {got}", "OK" if got == expected else "SAI")
    assert got == expected

quay lui('{a,b}c{d,e}f') = ['acdf', 'acef', 'bcdf', 'bcef'] OK
quay lui('abcd') = ['abcd'] OK


**Code nộp LeetCode (Cách 3):**

In [10]:
class Solution:
    def expand(self, s: str) -> List[str]:
        groups = []                                      # groups[k]: các lựa chọn của vị trí k (sắp xếp tăng)
        i = 0
        while i < len(s):
            if s[i] == "{":
                j = s.find("}", i)
                groups.append(sorted(s[i + 1:j].split(",")))
                i = j + 1                                # nhảy qua dấu }
            else:
                groups.append([s[i]])                    # chữ cái đứng riêng: một lựa chọn
                i += 1
        res, cur = [], []

        def dfs(k: int) -> None:
            if k == len(groups):                         # đã chọn xong mọi vị trí
                res.append("".join(cur))
                return
            for ch in groups[k]:                         # lựa chọn tăng dần: kết quả tự đúng thứ tự từ điển
                cur.append(ch)
                dfs(k + 1)
                cur.pop()

        dfs(0)
        return res

sol = lambda s: Solution().expand(s)
for s, expected in examples:
    assert sol(s) == expected
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** thời gian `O(W * L)` (mỗi từ được ghép một lần; bỏ qua `O(L log L)` sắp xếp các nhóm), bộ nhớ `O(L)` ngăn xếp đệ quy và `O(W * L)` cho kết quả. Độ sâu đệ quy chỉ `<= 50`.

---

### 7.3. Cách 4 — Xây theo từng tầng (không đệ quy)

**Ý tưởng khác.** Giữ danh sách `words` các từ đã xây xong cho các nhóm đầu; với nhóm tiếp theo (đã sắp xếp), thay `words` bằng `[w + ch for w in words for ch in group]`. Vòng ngoài duyệt từ cũ (đã đúng thứ tự), vòng trong duyệt lựa chọn tăng dần nên thứ tự từ điển được giữ. Không đệ quy.

In [11]:
def expand_levels(s: str) -> List[str]:
    words = [""]
    i = 0
    while i < len(s):
        if s[i] == "{":
            j = s.find("}", i)
            group = sorted(s[i + 1:j].split(","))
            i = j + 1
        else:
            group = [s[i]]
            i += 1
        words = [w + ch for w in words for ch in group]
    return words

for s, expected in examples:
    got = expand_levels(s)
    print(f"từng tầng({s!r}) = {got}", "OK" if got == expected else "SAI")
    assert got == expected

từng tầng('{a,b}c{d,e}f') = ['acdf', 'acef', 'bcdf', 'bcef'] OK
từng tầng('abcd') = ['abcd'] OK


**Code nộp LeetCode (Cách 4):**

In [12]:
class Solution:
    def expand(self, s: str) -> List[str]:
        words = [""]                                     # các từ đã xây cho các nhóm đầu (đã đúng thứ tự)
        i = 0
        while i < len(s):
            if s[i] == "{":
                j = s.find("}", i)
                group = sorted(s[i + 1:j].split(","))    # lựa chọn tăng dần
                i = j + 1
            else:
                group = [s[i]]
                i += 1
            words = [w + ch for w in words for ch in group]   # từ cũ ở vòng ngoài, lựa chọn mới ở vòng trong
        return words

sol_levels = lambda s: Solution().expand(s)
for s, expected in examples:
    assert sol_levels(s) == expected
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** tổng số phần tử tạo ra qua các tầng `<= 2W` (tầng cuối `W`, các tầng trước nhỏ hơn hoặc bằng một nửa với nhóm từ hai lựa chọn trở lên); thời gian `O(W * L)`, bộ nhớ `O(W * L)`.

## 8. TRACE từng bước

### 8.1. Phân tích nhóm cho `"{a,b}c{d,e}f"`

In từng bước đọc chuỗi và nhóm được tạo.

In [13]:
def trace_parse(s):
    groups, i = [], 0
    while i < len(s):
        if s[i] == "{":
            j = s.find("}", i)
            raw = s[i + 1:j]
            group = sorted(raw.split(","))
            print(f"i={i}: gặp '{{' -> '}}' ở {j}, phần giữa {raw!r} -> nhóm {group}")
            groups.append(group)
            i = j + 1
        else:
            print(f"i={i}: chữ cái {s[i]!r} -> nhóm [{s[i]!r}]")
            groups.append([s[i]])
            i += 1
    return groups

groups = trace_parse("{a,b}c{d,e}f")
print("groups =", groups)
assert groups == [["a", "b"], ["c"], ["d", "e"], ["f"]]

i=0: gặp '{' -> '}' ở 4, phần giữa 'a,b' -> nhóm ['a', 'b']
i=5: chữ cái 'c' -> nhóm ['c']
i=6: gặp '{' -> '}' ở 10, phần giữa 'd,e' -> nhóm ['d', 'e']
i=11: chữ cái 'f' -> nhóm ['f']
groups = [['a', 'b'], ['c'], ['d', 'e'], ['f']]


### 8.2. Quay lui cho `"{a,b}c{d,e}f"`

In từng lần chọn, vào sâu và hoàn tác.

In [14]:
def trace_backtrack(groups):
    res, cur = [], []
    def dfs(k, depth=0):
        pad = "  " * depth
        if k == len(groups):
            res.append("".join(cur))
            print(f"{pad}-> ghi nhận {res[-1]!r}")
            return
        for ch in groups[k]:
            cur.append(ch)
            print(f"{pad}nhóm {k}: chọn {ch!r}, cur = {''.join(cur)!r}")
            dfs(k + 1, depth + 1)
            cur.pop()
    dfs(0)
    return res

assert trace_backtrack([["a", "b"], ["c"], ["d", "e"], ["f"]]) == ["acdf", "acef", "bcdf", "bcef"]

nhóm 0: chọn 'a', cur = 'a'
  nhóm 1: chọn 'c', cur = 'ac'
    nhóm 2: chọn 'd', cur = 'acd'
      nhóm 3: chọn 'f', cur = 'acdf'
        -> ghi nhận 'acdf'
    nhóm 2: chọn 'e', cur = 'ace'
      nhóm 3: chọn 'f', cur = 'acef'
        -> ghi nhận 'acef'
nhóm 0: chọn 'b', cur = 'b'
  nhóm 1: chọn 'c', cur = 'bc'
    nhóm 2: chọn 'd', cur = 'bcd'
      nhóm 3: chọn 'f', cur = 'bcdf'
        -> ghi nhận 'bcdf'
    nhóm 2: chọn 'e', cur = 'bce'
      nhóm 3: chọn 'f', cur = 'bcef'
        -> ghi nhận 'bcef'


### 8.3. Xây theo từng tầng cho `"a{c,b}{e,d}"`

Nhóm có lựa chọn không sắp xếp sẵn (`c,b`): ta sắp xếp nhóm trước rồi mới ghép.

In [15]:
s = "a{c,b}{e,d}"
words, i = [""], 0
while i < len(s):
    if s[i] == "{":
        j = s.find("}", i)
        group = sorted(s[i + 1:j].split(","))
        i = j + 1
    else:
        group = [s[i]]
        i += 1
    words = [w + ch for w in words for ch in group]
    print(f"nhóm {group}: words = {words}")
assert words == ["abd", "abe", "acd", "ace"] == expand_recursive(s)

nhóm ['a']: words = ['a']
nhóm ['b', 'c']: words = ['ab', 'ac']
nhóm ['d', 'e']: words = ['abd', 'abe', 'acd', 'ace']


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (quay lui theo nhóm đã sắp xếp): kết quả tự đúng thứ tự, không cần sắp xếp cuối.

```python
class Solution:
    def expand(self, s: str) -> List[str]:
        groups = []                                      # groups[k]: các lựa chọn của vị trí k (sắp xếp tăng)
        i = 0
        while i < len(s):
            if s[i] == "{":
                j = s.find("}", i)
                groups.append(sorted(s[i + 1:j].split(",")))
                i = j + 1                                # nhảy qua dấu }
            else:
                groups.append([s[i]])                    # chữ cái đứng riêng: một lựa chọn
                i += 1
        res, cur = [], []

        def dfs(k: int) -> None:
            if k == len(groups):                         # đã chọn xong mọi vị trí
                res.append("".join(cur))
                return
            for ch in groups[k]:                         # lựa chọn tăng dần: kết quả tự đúng thứ tự từ điển
                cur.append(ch)
                dfs(k + 1)
                cur.pop()

        dfs(0)
        return res
```

In [16]:
class Solution:
    def expand(self, s: str) -> List[str]:
        groups = []                                      # groups[k]: các lựa chọn của vị trí k (sắp xếp tăng)
        i = 0
        while i < len(s):
            if s[i] == "{":
                j = s.find("}", i)
                groups.append(sorted(s[i + 1:j].split(",")))
                i = j + 1                                # nhảy qua dấu }
            else:
                groups.append([s[i]])                    # chữ cái đứng riêng: một lựa chọn
                i += 1
        res, cur = [], []

        def dfs(k: int) -> None:
            if k == len(groups):                         # đã chọn xong mọi vị trí
                res.append("".join(cur))
                return
            for ch in groups[k]:                         # lựa chọn tăng dần: kết quả tự đúng thứ tự từ điển
                cur.append(ch)
                dfs(k + 1)
                cur.pop()

        dfs(0)
        return res

sol = lambda s: Solution().expand(s)
for s, expected in examples:
    got = sol(s)
    print(got, "OK" if got == expected else "SAI")
    assert got == expected

['acdf', 'acef', 'bcdf', 'bcef'] OK
['abcd'] OK


## 10. Vì sao đúng

**Mô hình.** `s` là ghép của `m` **nhóm** theo thứ tự xuất hiện: mỗi chữ cái đứng riêng là nhóm một phần tử, mỗi cặp ngoặc `{x,y,...}` là nhóm gồm các chữ cái trong đó (các chữ cái trong một cặp ngoặc đôi một khác nhau). Một từ hợp lệ là chọn **đúng một** phần tử trong mỗi nhóm rồi ghép theo thứ tự nhóm. Vì mọi lựa chọn là **một chữ cái**, mọi từ có đúng `m` chữ cái. Hai bộ lựa chọn khác nhau cho hai từ khác nhau (khác nhau ở một vị trí chữ cái nào đó), nên không có trùng lặp và số từ bằng tích kích thước các nhóm.

**Thứ tự từ điển.** Xét hai từ `u != v` cùng độ dài `m`. Vị trí khác nhau đầu tiên là `k`: `u < v` khi và chỉ khi `u[k] < v[k]`. Thuật toán quay lui duyệt nhóm `0` ở vòng ngoài nhất, trong mỗi lựa chọn của nhóm `0` duyệt nhóm `1`,... Với lựa chọn trong mỗi nhóm đã sắp xếp tăng, từ `u` được sinh ra trước `v` khi và chỉ khi tại nhóm đầu tiên mà hai bộ lựa chọn khác nhau, `u` chọn chữ cái đứng trước (nhỏ hơn). Đó đúng là điều kiện `u < v`. Vậy thứ tự sinh ra trùng với thứ tự từ điển.

**Cách 2** dùng `product` và `sorted` nên đúng chắc chắn. **Cách 4** là cùng thứ tự: sau mỗi tầng `words` được sắp xếp (quy nạp: nếu `words` đã sắp xếp và có cùng độ dài thì `[w + ch for w in words for ch in group]` cũng đã sắp xếp vì so sánh theo `w` trước, rồi theo `ch`). **Cách 1** sinh mọi khả năng bằng đệ quy trên hậu tố và sắp xếp cuối cùng nên không phụ thuộc lập luận trên: chính nó là **oracle**.

**Kiểm chứng bằng máy.** Ta đối chiếu mọi cách với oracle (Cách 1) trên **mọi** chuỗi hợp lệ ghép từ các "mảnh" (chữ cái đứng riêng và nhóm ngoặc có `1..3` chữ cái đôi một khác nhau, theo mọi thứ tự lựa chọn) tới `3` mảnh trên bảng chữ cái `{a, b, c}`, cộng nhiều chuỗi ngẫu nhiên tới `50` ký tự.

In [17]:
pieces = ["a", "b", "c"] + ["{" + ",".join(p) + "}" for k in (1, 2, 3) for p in permutations("abc", k)]
print("số mảnh cơ bản:", len(pieces), "(chữ cái đứng riêng và các nhóm ngoặc theo mọi thứ tự lựa chọn)")
impls = {"đệ quy hậu tố": expand_recursive, "product": expand_product, "quay lui": expand_backtrack, "từng tầng": expand_levels,
         "nộp (Cách 3)": sol, "Cách 2 (nộp)": sol_product, "Cách 4 (nộp)": sol_levels}
total = 0
for count in range(1, 4):
    for combo in product(pieces, repeat=count):
        s = "".join(combo)
        ref = expand_recursive(s)
        for name, fn in impls.items():
            assert fn(s) == ref, (name, s)
        total += 1
print("đã đối chiếu", total, "chuỗi ghép từ 1..3 mảnh: mọi cách khớp oracle")

số mảnh cơ bản: 18 (chữ cái đứng riêng và các nhóm ngoặc theo mọi thứ tự lựa chọn)


đã đối chiếu 6174 chuỗi ghép từ 1..3 mảnh: mọi cách khớp oracle


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Đệ quy hậu tố + sort | `O(W * L + W log W)` | `O(W * L)` |
| 2. `product` + sort | `O(W * L + W log W)` | `O(W * L)` |
| 3. Quay lui đã sắp xếp | `O(W * L)` | `O(L)` + kết quả |
| 4. Xây từng tầng | `O(W * L)` | `O(W * L)` |

(`W` là số từ, tối đa vài nghìn với `s` dài `50`; `L` là số nhóm.) Đo ở cỡ tối đa với các dạng: nhiều nhóm `3` lựa chọn, nhiều nhóm `2` lựa chọn, chữ cái đứng riêng, nhóm `5` lựa chọn:

In [18]:
cases = {
    "{a,b,c} x 7 + d (độ dài 50)": "{a,b,c}" * 7 + "d",
    "{a,b} x 10 (độ dài 50)": "{a,b}" * 10,
    "{a,b,c,d,e} x 4 (độ dài 44)": "{a,b,c,d,e}" * 4,
    "chữ cái đứng riêng (độ dài 50)": (string.ascii_lowercase * 2)[:50],
    "xen kẽ: a{b,c}d{e,f,g} ...": "a{b,c}d{e,f,g}" * 3 + "h",
    "nhóm 17 chữ + ab": "{" + ",".join(string.ascii_lowercase[:17]) + "}" + "ab",
}
for label, s in cases.items():
    assert len(s) <= 50, (label, len(s))
    ref = sol(s)
    row = [f"{label:34}"]
    for name, fn in (("quay lui", sol), ("product", sol_product), ("từng tầng", sol_levels), ("đệ quy hậu tố", expand_recursive)):
        t0 = time.perf_counter()
        got = fn(s)
        dt = (time.perf_counter() - t0) * 1000
        assert got == ref, (label, name)
        row.append(f"{name} {dt:7.2f} ms")
    print(" | ".join(row), "| số từ", len(ref))

{a,b,c} x 7 + d (độ dài 50)        | quay lui    0.77 ms | product    0.76 ms | từng tầng    0.85 ms | đệ quy hậu tố    0.22 ms | số từ 2187
{a,b} x 10 (độ dài 50)             | quay lui    0.33 ms | product    0.29 ms | từng tầng    0.29 ms | đệ quy hậu tố    0.09 ms | số từ 1024
{a,b,c,d,e} x 4 (độ dài 44)        | quay lui    0.11 ms | product    0.10 ms | từng tầng    0.10 ms | đệ quy hậu tố    0.05 ms | số từ 625
chữ cái đứng riêng (độ dài 50)     | quay lui    0.02 ms | product    0.01 ms | từng tầng    0.01 ms | đệ quy hậu tố    0.02 ms | số từ 1
xen kẽ: a{b,c}d{e,f,g} ...         | quay lui    0.10 ms | product    0.10 ms | từng tầng    0.18 ms | đệ quy hậu tố    0.05 ms | số từ 216
nhóm 17 chữ + ab                   | quay lui    0.01 ms | product    0.01 ms | từng tầng    0.01 ms | đệ quy hậu tố    0.01 ms | số từ 17


## 12. Bộ test

Đáp án là danh sách đã **sắp xếp**, nên so sánh trực tiếp với oracle; thêm một **hàm kiểm tra độc lập** (`is_valid_output`): danh sách tăng chặt, số từ bằng tích kích thước các nhóm, và mỗi từ khớp mẫu (từng vị trí thuộc tập lựa chọn của nhóm tương ứng). Bộ test gồm: ví dụ của đề, edge case có kỳ vọng viết tay, kiểm tra hàm kiểm tra từ chối kết quả sai, phản ví dụ cho các cách sai (tìm bằng tìm kiếm), test ngẫu nhiên có seed (đúng ràng buộc) và tính chất.

In [19]:
def option_sets(s):
    """Tập lựa chọn của từng vị trí, dựng bằng cách quét từng ký tự (khác cách tách nhóm ở trên)."""
    sets, inside, cur = [], False, set()
    for ch in s:
        if ch == "{":
            inside, cur = True, set()
        elif ch == "}":
            inside = False
            sets.append(cur)
        elif ch == ",":
            continue
        elif inside:
            cur.add(ch)
        else:
            sets.append({ch})
    return sets

def is_valid_output(s: str, res: List[str]) -> bool:
    sets = option_sets(s)
    size = 1
    for st in sets:
        size *= len(st)
    if len(res) != size or any(a >= b for a, b in zip(res, res[1:])):       # đủ số từ và tăng chặt
        return False
    return all(len(w) == len(sets) and all(w[k] in sets[k] for k in range(len(sets))) for w in res)

def check_all(s, expected=None, brute=True):
    results = {name: fn(s) for name, fn in impls.items()}
    first = results["nộp (Cách 3)"]
    for name, got in results.items():
        assert got == first, (name, s, got[:5], first[:5])
    assert is_valid_output(s, first), s
    if brute:
        assert expand_recursive(s) == first, s
    if expected is not None:
        assert first == expected, (s, first, expected)
    return first

# 12.1 ví dụ đề
for s, e in examples:
    check_all(s, e)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ("a", ["a"]),                                          # một chữ cái
    ("abc", ["abc"]),                                      # không có ngoặc
    ("{a}", ["a"]),                                        # nhóm một lựa chọn trong ngoặc
    ("{a,b}", ["a", "b"]),                                 # chỉ một nhóm
    ("{b,a}", ["a", "b"]),                                 # lựa chọn không theo thứ tự: phải sắp xếp
    ("{a,b}{c,d}", ["ac", "ad", "bc", "bd"]),              # hai nhóm liền nhau
    ("a{b,c}d", ["abd", "acd"]),                           # nhóm ở giữa
    ("{c,b,a}", ["a", "b", "c"]),                          # đảo ngược hoàn toàn
    ("a{c,b}{e,d}", ["abd", "abe", "acd", "ace"]),         # hai nhóm, đều ngược thứ tự
    ("{a,b}{a,b}", ["aa", "ab", "ba", "bb"]),              # hai nhóm cùng tập lựa chọn: mỗi vị trí độc lập
    ("{z,a}z", ["az", "zz"]),                              # chữ cái đứng riêng đứng sau
    ("z{z,a}", ["za", "zz"]),                              # chữ cái đứng riêng đứng trước
    ("{a,b,c}{d,e}", ["ad", "ae", "bd", "be", "cd", "ce"]),# 3 x 2 từ
    ("{c,a}b{e,d}", ["abd", "abe", "cbd", "cbe"]),         # tính tay ở bài tập 2
    ("{a,b}" * 2, ["aa", "ab", "ba", "bb"]),
]
for s, e in edge:
    check_all(s, e)
big = check_all("{a,b,c}" * 7 + "d")
assert len(big) == 3 ** 7 and big[0] == "aaaaaaad" and big[-1] == "cccccccd"       # độ dài tối đa
print("12.2 edge case: OK", len(edge) + 1, "ca")

# 12.3 hàm kiểm tra phải từ chối kết quả sai
assert is_valid_output("{b,a}c", ["ac", "bc"])
assert not is_valid_output("{b,a}c", ["bc", "ac"])              # không sắp xếp
assert not is_valid_output("{b,a}c", ["ac"])                    # thiếu từ
assert not is_valid_output("{b,a}c", ["ac", "ac"])              # trùng lặp
assert not is_valid_output("{b,a}c", ["ac", "bd"])              # từ không khớp
print("12.3 hàm kiểm tra: OK")

12.1 ví dụ đề: OK
12.2 edge case: OK 16 ca
12.3 hàm kiểm tra: OK


In [20]:
# 12.4 phản ví dụ: các cách sai (tìm bằng tìm kiếm so với oracle)
def wrong_no_sort(s):                              # quên sắp xếp lựa chọn của nhóm và kết quả
    groups, i = [], 0
    while i < len(s):
        if s[i] == "{":
            j = s.find("}", i)
            groups.append(s[i + 1:j].split(","))
            i = j + 1
        else:
            groups.append([s[i]])
            i += 1
    return ["".join(t) for t in product(*groups)]

def wrong_comma_option(s):                         # coi dấu phẩy là một lựa chọn (đọc mọi ký tự trong ngoặc)
    groups, i = [], 0
    while i < len(s):
        if s[i] == "{":
            j = s.find("}", i)
            groups.append(sorted(s[i + 1:j]))      # thiếu split(","): có cả ','
            i = j + 1
        else:
            groups.append([s[i]])
            i += 1
    return sorted("".join(t) for t in product(*groups))

def wrong_keep_brace(s):                           # quên nhảy qua dấu '}' (i = j thay vì j + 1)
    groups, i = [], 0
    while i < len(s):
        if s[i] == "{":
            j = s.find("}", i)
            groups.append(sorted(s[i + 1:j].split(",")))
            i = j                                  # SAI: lần sau gặp '}' và coi nó là chữ cái đứng riêng
        else:
            groups.append([s[i]])
            i += 1
    return sorted("".join(t) for t in product(*groups))

def wrong_string_group(s):                         # nhóm chữ cái đứng riêng là CHUỖI "abc" thay vì danh sách ["a"], ["b"], ["c"]
    groups, i = [], 0
    while i < len(s):
        if s[i] == "{":
            j = s.find("}", i)
            groups.append(sorted(s[i + 1:j].split(",")))
            i = j + 1
        else:
            j = i
            while j < len(s) and s[j] != "{":
                j += 1
            groups.append(s[i:j])                  # SAI: cả đoạn chữ cái đứng riêng thành một "nhóm" chuỗi; product tách thành lựa chọn
            i = j
    return sorted("".join(t) for t in product(*groups))

def wrong_reverse(s):                              # ghép ngược: lựa chọn mới ghép vào đầu
    words, i = [""], 0
    while i < len(s):
        if s[i] == "{":
            j = s.find("}", i)
            group = sorted(s[i + 1:j].split(","))
            i = j + 1
        else:
            group = [s[i]]
            i += 1
        words = [ch + w for w in words for ch in group]
    return sorted(words)

def random_pattern(rng, pieces_count, letters="abcde"):
    out = []
    for _ in range(pieces_count):
        if rng.random() < 0.5:
            out.append(rng.choice(letters))
        else:
            k = rng.randint(1, min(3, len(letters)))
            out.append("{" + ",".join(rng.sample(letters, k)) + "}")
    return "".join(out)

def find_counter(wrong, tries=20000, seed=9):
    rng = random.Random(seed)
    best = None
    for _ in range(tries):
        s = random_pattern(rng, rng.randint(1, 4))
        if best is not None and len(s) >= len(best):
            continue
        try:
            bad = wrong(s) != expand_recursive(s)
        except Exception:
            bad = True
        if bad:
            best = s
    return best

hand_cases = {                                       # phản ví dụ chọn tay (dễ nhìn), được xác nhận lại bằng oracle
    "quên sắp xếp": (wrong_no_sort, "{b,a}c"),
    "coi dấu phẩy là một lựa chọn": (wrong_comma_option, "{a,b}c"),
    "quên nhảy qua dấu }": (wrong_keep_brace, "{a,b}c"),
    "biểu diễn chữ cái đứng riêng bằng chuỗi": (wrong_string_group, "abcd"),
    "ghép ngược thứ tự": (wrong_reverse, "a{b,c}d"),
}
counters = {}
for name, (fn, s) in hand_cases.items():
    wrong_result = fn(s)
    right = expand_recursive(s)
    assert wrong_result != right, name
    assert find_counter(fn) is not None, name           # tìm kiếm tự động cũng tìm ra phản ví dụ (nhỏ nhất)
    counters[name] = s
    print(f"{name:42} s={s!r}: sai -> {wrong_result}, đúng -> {right}")
    check_all(s)
print("12.4 phản ví dụ: OK")

quên sắp xếp                               s='{b,a}c': sai -> ['bc', 'ac'], đúng -> ['ac', 'bc']


coi dấu phẩy là một lựa chọn               s='{a,b}c': sai -> [',c', 'ac', 'bc'], đúng -> ['ac', 'bc']


quên nhảy qua dấu }                        s='{a,b}c': sai -> ['a}c', 'b}c'], đúng -> ['ac', 'bc']


biểu diễn chữ cái đứng riêng bằng chuỗi    s='abcd': sai -> ['a', 'b', 'c', 'd'], đúng -> ['abcd']


ghép ngược thứ tự                          s='a{b,c}d': sai -> ['dba', 'dca'], đúng -> ['abd', 'acd']
12.4 phản ví dụ: OK


In [21]:
# 12.5 ngẫu nhiên nhỏ: nhiều bảng chữ cái, thứ tự lựa chọn ngẫu nhiên
rng = random.Random(4087)
for _ in range(3000):
    check_all(random_pattern(rng, rng.randint(1, 6), rng.choice(["ab", "abc", "abcde", string.ascii_lowercase])))
print("12.5 ngẫu nhiên nhỏ: OK")

# 12.6 ngẫu nhiên tới độ dài 50 (so các cách; oracle khi số từ vừa phải)
for _ in range(200):
    parts, length, count = [], 0, 1
    while True:
        if rng.random() < 0.5:
            piece, size = rng.choice(string.ascii_lowercase), 1
        else:
            size = rng.randint(1, 4)
            piece = "{" + ",".join(rng.sample(string.ascii_lowercase, size)) + "}"
        if length + len(piece) > 50 or count * size > 3000:
            break
        parts.append(piece)
        length += len(piece)
        count *= size
    s = "".join(parts) or "a"
    check_all(s, brute=count <= 1500)
print("12.6 ngẫu nhiên tới độ dài 50: OK")

# 12.7 tính chất: số từ = tích kích thước nhóm; không trùng; thêm chữ cái đứng riêng ở cuối thêm hậu tố; đổi thứ tự lựa chọn trong ngoặc không đổi kết quả
for _ in range(500):
    s = random_pattern(rng, rng.randint(1, 5), "abcdef")
    res = sol(s)
    sizes = [len(x) for x in option_sets(s)]
    prod = 1
    for z in sizes:
        prod *= z
    assert len(res) == prod == len(set(res))
    assert sol(s + "z") == [w + "z" for w in res]                                     # thêm chữ cái đứng riêng
    shuffled = "".join(("{" + ",".join(rng.sample(sorted(st), len(st))) + "}") if len(st) > 1 else next(iter(st)) for st in option_sets(s))
    assert sol(shuffled) == res                                                         # đổi thứ tự lựa chọn trong ngoặc
print("12.7 tính chất: OK")

12.5 ngẫu nhiên nhỏ: OK


12.6 ngẫu nhiên tới độ dài 50: OK
12.7 tính chất: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên sắp xếp

Đề yêu cầu danh sách **đã sắp xếp** theo thứ tự từ điển; các lựa chọn trong ngoặc có thể xuất hiện không theo thứ tự.

In [22]:
s = counters["quên sắp xếp"]
print(f"s={s!r} | sai ->", wrong_no_sort(s), "| đúng ->", sol(s))
assert wrong_no_sort(s) != sol(s)

s='{b,a}c' | sai -> ['bc', 'ac'] | đúng -> ['ac', 'bc']


Vì sao sai: `"{b,a}c"` cho `["bc", "ac"]` nếu giữ nguyên thứ tự trong ngoặc. **Đúng:** `sorted` các lựa chọn của mỗi nhóm (hoặc sắp xếp kết quả cuối cùng).

### Lỗi 2 — Coi dấu phẩy là một lựa chọn

Nếu lấy mọi ký tự trong ngoặc mà quên tách theo dấu phẩy thì `','` trở thành một lựa chọn.

In [23]:
s = counters["coi dấu phẩy là một lựa chọn"]
print(f"s={s!r} | sai ->", wrong_comma_option(s), "| đúng ->", sol(s))
assert wrong_comma_option(s) != sol(s)

s='{a,b}c' | sai -> [',c', 'ac', 'bc'] | đúng -> ['ac', 'bc']


Vì sao sai: dấu phẩy chỉ là dấu ngăn cách. **Đúng:** `s[i + 1:j].split(",")`.

### Lỗi 3 — Quên nhảy qua dấu `}`

Đặt `i = j` thay vì `i = j + 1` làm vòng lặp sau coi `}` là một chữ cái đứng riêng.

In [24]:
s = counters["quên nhảy qua dấu }"]
print(f"s={s!r} | sai ->", wrong_keep_brace(s), "| đúng ->", sol(s))
assert wrong_keep_brace(s) != sol(s)

s='{a,b}c' | sai -> ['a}c', 'b}c'] | đúng -> ['ac', 'bc']


Vì sao sai: kết quả có thêm ký tự `}` ở giữa mỗi từ. **Đúng:** sau khi xử lý nhóm, `i = j + 1`.

### Lỗi 4 — Biểu diễn chữ cái đứng riêng bằng chuỗi rồi đưa vào `product`

`product("abcd")` coi chuỗi là dãy các lựa chọn `'a'`, `'b'`, `'c'`, `'d'`, không phải một nhóm cố định.

In [25]:
s = counters["biểu diễn chữ cái đứng riêng bằng chuỗi"]
print(f"s={s!r} | sai ->", wrong_string_group(s), "| đúng ->", sol(s))
assert wrong_string_group(s) != sol(s)

s='abcd' | sai -> ['a', 'b', 'c', 'd'] | đúng -> ['abcd']


Vì sao sai: mỗi nhóm phải là **danh sách các lựa chọn**; chữ cái đứng riêng là danh sách một phần tử `["a"]`. **Đúng:** `groups.append([s[i]])`.

### Lỗi 5 — Ghép ngược thứ tự

In [26]:
s = counters["ghép ngược thứ tự"]
print(f"s={s!r} | sai ->", wrong_reverse(s), "| đúng ->", sol(s))
assert wrong_reverse(s) != sol(s)

s='a{b,c}d' | sai -> ['dba', 'dca'] | đúng -> ['abd', 'acd']


Vì sao sai: lựa chọn của nhóm đứng sau phải nằm **sau** trong từ. **Đúng:** `w + ch` (từ cũ trước, lựa chọn mới sau).

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "mỗi vị trí có một vài lựa chọn, liệt kê mọi tổ hợp" ⇒ **tích Descartes** (backtracking hoặc `itertools.product`).
- Muốn kết quả đã **sắp xếp** từ điển: sắp xếp các lựa chọn trong từng vị trí, duyệt vị trí đầu ở vòng ngoài nhất; mọi từ cùng độ dài.
- Mẫu backtracking: `chọn → đi sâu → hoàn tác`.
- Phân tích cú pháp đơn giản: quét một lượt bằng chỉ số, `find("}")` tìm cặp ngoặc, `split(",")` tách lựa chọn.
- Cẩn thận `product` với **chuỗi** (nó tách thành chữ cái).
- Cùng họ: Letter Combinations of a Phone Number (17), Generate Parentheses (22), Brace Expansion II (1096), Permutations (46).
- Test nên có: không có ngoặc, một nhóm, lựa chọn đảo thứ tự, nhóm một phần tử trong ngoặc, độ dài `50` với nhiều nhóm `3` lựa chọn.

## 15. Biến thể

1. **Lựa chọn nhiều chữ cái** (ví dụ `"{ab,c}d"`): từ có độ dài khác nhau; thứ tự từ điển không còn đảm bảo bằng thứ tự sinh nên phải `sorted` cuối và (nếu có trùng) loại trùng.
2. **Brace Expansion II (1096):** có lồng nhau và phép hợp/tích.
3. **Đếm số từ** hoặc **từ thứ `k`**: dùng cơ số hỗn hợp (bài tập 5).
4. Bài liên quan: 17, 22, 1096.

Kiểm biến thể 1 (lựa chọn nhiều chữ cái, có thể trùng) bằng oracle đệ quy trên hậu tố:

In [27]:
def expand_general(s: str) -> List[str]:
    """Các lựa chọn có thể dài nhiều chữ cái và có thể trùng: ghép bằng tập rồi sắp xếp."""
    groups, i = [], 0
    while i < len(s):
        if s[i] == "{":
            j = s.find("}", i)
            groups.append(sorted(set(s[i + 1:j].split(","))))
            i = j + 1
        else:
            groups.append([s[i]])                          # chữ cái đứng riêng: nhóm một phần tử
            i += 1
    words = {""}
    for group in groups:
        words = {w + o for w in words for o in group}
    return sorted(words)

def expand_general_oracle(s: str) -> List[str]:
    def rec(t):
        if not t:
            return {""}
        if t[0] == "{":
            j = t.find("}")
            return {o + r for o in t[1:j].split(",") for r in rec(t[j + 1:])}
        return {t[0] + r for r in rec(t[1:])}
    return sorted(rec(s))

rng = random.Random(15)
for _ in range(1500):
    parts = []
    for _ in range(rng.randint(1, 4)):
        if rng.random() < 0.5:
            parts.append(rng.choice("abc"))
        else:
            parts.append("{" + ",".join("".join(rng.choice("abc") for _ in range(rng.randint(1, 2))) for _ in range(rng.randint(1, 3))) + "}")
    s = "".join(parts)
    assert expand_general(s) == expand_general_oracle(s), s
assert expand_general("{ab,c}d") == ["abd", "cd"]
print("biến thể lựa chọn nhiều chữ cái: tập + sắp xếp cuối khớp oracle; ví dụ", expand_general("{ab,c}d{e,e}"))

biến thể lựa chọn nhiều chữ cái: tập + sắp xếp cuối khớp oracle; ví dụ ['abde', 'cde']


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đệ quy hậu tố + sort | Theo định nghĩa | `O(W L + W log W)` | `O(W L)` | Oracle |
| 2. `product` + sort | Tách nhóm, tích Descartes | `O(W L + W log W)` | `O(W L)` | Ngắn gọn |
| 3. Quay lui đã sắp xếp | Không cần sort cuối | `O(W L)` | `O(L)` + kết quả | **Cách nộp chính** |
| 4. Xây từng tầng | Không đệ quy | `O(W L)` | `O(W L)` | Khi muốn tránh đệ quy |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán rồi chạy để kiểm tra:

```python
print(parse_groups("a{b,c}d"))
print("{x,y}".find("}"), "a,b,c".split(","))
print(Solution().expand("{b,a}c"))
```

In [28]:
# Bài 1: ghi dự đoán của bạn
my_guess_1 = None      # các nhóm của "a{b,c}d"
my_guess_2 = None      # (vị trí }, danh sách tách)
my_guess_3 = None      # kết quả expand

### Bài 2 — Tính tay (dễ-vừa)

`s = "{c,a}b{e,d}"`. Liệt kê các nhóm (đã sắp xếp) và các từ theo thứ tự.

In [29]:
# Bài 2: điền các nhóm và danh sách từ
hand_groups, hand_answer = None, None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành quay lui.

In [30]:
def exercise_3(s: str) -> List[str]:
    groups, i = [], 0
    while i < len(s):
        if s[i] == "{":
            j = s.find("}", i)
            # TODO 1: thêm nhóm gồm các lựa chọn đã sắp xếp: sorted(s[i + 1:j].split(","))
            # TODO 2: đặt i = j + 1
            pass
        else:
            groups.append([s[i]])
            i += 1
    res, cur = [], []

    def dfs(k):
        if k == len(groups):
            res.append("".join(cur))
            return
        for ch in groups[k]:
            # TODO 3: cur.append(ch); dfs(k + 1); cur.pop()
            pass

    dfs(0)
    return res


# for s, e in examples:
#     assert exercise_3(s) == e

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 4, tự viết `exercise_4(s)` bằng **xây từng tầng** (danh sách `words`).

In [31]:
def exercise_4(s: str) -> List[str]:
    # TODO
    pass


# for s, e in examples:
#     assert exercise_4(s) == e

### Bài 5 — Mini challenge: từ thứ k mà không sinh hết (khó)

Viết `exercise_5(s, k)` trả về từ **thứ `k`** (đếm từ `0`) trong danh sách đã sắp xếp mà **không sinh ra toàn bộ danh sách**. Ví dụ với `s = "{a,b}c{d,e}f"`: `exercise_5(s, 0) = "acdf"`, `exercise_5(s, 3) = "bcef"`. Có thể giả sử `0 <= k < số từ`.

Gợi ý: coi `k` là số viết theo **cơ số hỗn hợp**, nhóm cuối là chữ số hàng đơn vị: lặp từ nhóm cuối về đầu, `k, d = divmod(k, len(nhóm))` rồi lấy `nhóm[d]`.

In [32]:
def exercise_5(s: str, k: int) -> str:
    # TODO
    pass


# assert exercise_5("{a,b}c{d,e}f", 0) == "acdf"
# assert exercise_5("{a,b}c{d,e}f", 3) == "bcef"
# assert exercise_5("{b,a}c", 1) == "bc"

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: chữ cái đứng riêng là nhóm một phần tử.
- Mức 2: `"a{b,c}d"` có ba nhóm: `a`, `{b,c}`, `d`.
- Mức 3: `[['a'], ['b', 'c'], ['d']]`; `(4, ['a', 'b', 'c'])` (vị trí `}` trong `"{x,y}"` là `4`); `["ac", "bc"]`.

**Bài 2**
- Mức 1: các nhóm: `{c,a}`, `b`, `{e,d}`; sắp xếp từng nhóm.
- Mức 2: `[['a','c'], ['b'], ['d','e']]`; số từ `2 * 1 * 2 = 4`.
- Mức 3: `["abd", "abe", "cbd", "cbe"]`.

**Bài 3**
- Mức 1: `groups.append(sorted(s[i + 1:j].split(",")))`.
- Mức 2: `i = j + 1`.
- Mức 3: `cur.append(ch)`, `dfs(k + 1)`, `cur.pop()`.

**Bài 4**
- Mức 1: `words = [""]`.
- Mức 2: với mỗi nhóm đã sắp xếp: `words = [w + ch for w in words for ch in group]`.
- Mức 3: trả `words`.

**Bài 5**
- Mức 1: dựng `groups` đã sắp xếp như Cách 3.
- Mức 2: duyệt `groups` từ cuối về đầu: `k, d = divmod(k, len(group))`; lấy `group[d]`.
- Mức 3: ghép các chữ cái theo thứ tự nhóm (đảo ngược danh sách vừa thu được).

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [33]:
r1 = parse_groups("a{b,c}d")
r2 = ("{x,y}".find("}"), "a,b,c".split(","))
r3 = Solution().expand("{b,a}c")
print(r1, r2, r3)
assert r1 == [["a"], ["b", "c"], ["d"]]
assert r2 == (4, ["a", "b", "c"])
assert r3 == ["ac", "bc"]

[['a'], ['b', 'c'], ['d']] (4, ['a', 'b', 'c']) ['ac', 'bc']


### Đáp án Bài 2

Các nhóm đã sắp xếp: `[["a", "c"], ["b"], ["d", "e"]]`; `2 * 1 * 2 = 4` từ: `"abd"`, `"abe"`, `"cbd"`, `"cbe"`.

In [34]:
hand_groups_solution, hand_answer_solution = [["a", "c"], ["b"], ["d", "e"]], ["abd", "abe", "cbd", "cbe"]
assert hand_groups_solution == parse_groups("{c,a}b{e,d}")
assert hand_answer_solution == sol("{c,a}b{e,d}") == expand_recursive("{c,a}b{e,d}")
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [35]:
def exercise_3_solution(s: str) -> List[str]:
    groups, i = [], 0
    while i < len(s):
        if s[i] == "{":
            j = s.find("}", i)
            groups.append(sorted(s[i + 1:j].split(",")))
            i = j + 1
        else:
            groups.append([s[i]])
            i += 1
    res, cur = [], []

    def dfs(k):
        if k == len(groups):
            res.append("".join(cur))
            return
        for ch in groups[k]:
            cur.append(ch)
            dfs(k + 1)
            cur.pop()

    dfs(0)
    return res

for s, e in examples:
    assert exercise_3_solution(s) == e
rng = random.Random(3)
for _ in range(500):
    s = random_pattern(rng, rng.randint(1, 5))
    assert exercise_3_solution(s) == expand_recursive(s)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [36]:
def exercise_4_solution(s: str) -> List[str]:
    words, i = [""], 0
    while i < len(s):
        if s[i] == "{":
            j = s.find("}", i)
            group = sorted(s[i + 1:j].split(","))
            i = j + 1
        else:
            group = [s[i]]
            i += 1
        words = [w + ch for w in words for ch in group]
    return words

for s, e in examples:
    assert exercise_4_solution(s) == e
rng = random.Random(4)
for _ in range(500):
    s = random_pattern(rng, rng.randint(1, 5))
    assert exercise_4_solution(s) == expand_recursive(s)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [37]:
def exercise_5_solution(s: str, k: int) -> str:
    groups, i = [], 0
    while i < len(s):
        if s[i] == "{":
            j = s.find("}", i)
            groups.append(sorted(s[i + 1:j].split(",")))
            i = j + 1
        else:
            groups.append([s[i]])
            i += 1
    chars = []
    for group in reversed(groups):                 # nhóm cuối là chữ số hàng đơn vị của số cơ số hỗn hợp
        k, d = divmod(k, len(group))
        chars.append(group[d])
    return "".join(reversed(chars))

assert exercise_5_solution("{a,b}c{d,e}f", 0) == "acdf"
assert exercise_5_solution("{a,b}c{d,e}f", 3) == "bcef"
assert exercise_5_solution("{b,a}c", 1) == "bc"
rng = random.Random(5)
for _ in range(500):
    s = random_pattern(rng, rng.randint(1, 5))
    words = expand_recursive(s)
    for idx in (0, len(words) - 1, rng.randrange(len(words))):
        assert exercise_5_solution(s, idx) == words[idx], (s, idx)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (tách nhóm + quay lui)

```python
class Solution:
    def expand(self, s: str) -> List[str]:
        groups = []
        i = 0
        while i < len(s):
            if s[i] == "{":
                j = s.find("}", i)
                groups.append(sorted(s[i + 1:j].split(",")))
                i = j + 1
            else:
                groups.append([s[i]])
                i += 1
        res, cur = [], []

        def dfs(k):
            if k == len(groups):
                res.append("".join(cur))
                return
            for ch in groups[k]:
                cur.append(ch)
                dfs(k + 1)
                cur.pop()

        dfs(0)
        return res
```

Công thức nhớ nhanh: **tách nhóm → sắp xếp từng nhóm → quay lui theo thứ tự nhóm (kết quả tự đúng thứ tự từ điển)**.

### Checklist trước khi submit

- [ ] Mỗi nhóm là **danh sách** các lựa chọn (chữ cái đứng riêng là `[c]`, không phải chuỗi)?
- [ ] Lựa chọn trong ngoặc được tách bằng `split(",")` (không lấy dấu phẩy) và **sắp xếp**?
- [ ] Sau khi xử lý nhóm ngoặc, đã nhảy qua `}` (`i = j + 1`)?
- [ ] Ghép từ theo thứ tự nhóm (`cur.append` rồi `cur.pop`)?
- [ ] Đã thử: không có ngoặc, một nhóm, lựa chọn đảo thứ tự, `{a}`, độ dài `50` với `3^7` từ?